# LabPulse — Laboratory Operations Analytics

## Project overview
LabPulse is an exploratory data analysis project that examines synthetic laboratory result records to understand turnaround time (TAT), sample workload, and delay patterns.

The notebook uses Python and pandas for data preparation and analysis, with Matplotlib and Seaborn for visual exploration.

## Business context
Laboratory turnaround time is the elapsed time between specimen collection and result reporting. Understanding its distribution and variation across tests and collection periods can help identify patterns worth investigating.

## Objectives
- Explore laboratory result records and assess data quality.
- Calculate turnaround time in minutes.
- Compare TAT and sample workload across tests, weekdays, hours, and months.
- Explore records at or above an **180-minute exploratory threshold**.
- Summarize findings and export a processed dataset.

## Dataset and interpretation
The source is the **Auric Grid Synthetic Hospital Interoperability Environment** dataset. The data is synthetic and does not represent a real hospital, laboratory, patient population, or operational service-level agreement.

**Important:** The 180-minute threshold is an analytical classification selected for this project. It is not an official SLA or clinical standard.

In [ ]:

from pathlib import Path

import pandas as pd

# Load the dataset
data_path = Path("../data/raw/longitudinal.csv")
df = pd.read_csv(data_path, low_memory=False)

# Display first 5 records
print(df.head())

# Check number of rows and columns
print(df.shape)

# Display column names
print(df.columns.tolist())

# Check data types
print(df.dtypes)


## 2. Load and explore the source data

The following cells inspect the source dataset, event types, laboratory-related categories, and missingness before focusing on reported laboratory results.

In [ ]:

# Count different event types
event_counts = df['event_type'].value_counts()

# Display all event types
print(event_counts.to_string())

# Count the total number of unique event types
print("\nTotal event types:", df['event_type'].nunique())


In [ ]:

# Filter laboratory-related event categories
lab_df = df[
    df['event_category'].str.contains(
        'laboratory|lab',
        case=False,
        na=False
    )
]

print("Total laboratory event records:", len(lab_df))

print("\nLaboratory event types:")
print(lab_df['event_type'].value_counts().to_string())


In [ ]:

# Filter laboratory orders
lab_orders = df[
    df['event_type'] == 'lab_order_placed'
]

# Display the first 10 laboratory orders
print(lab_orders.head(10))

# Count unique laboratory orders
print(
    "Unique order IDs:",
    lab_orders['order_id'].nunique()
)


In [ ]:

# Count missing values in every column
missing_values = df.isnull().sum()

# Calculate missing value percentage
missing_percentage = (
    df.isnull().mean() * 100
).round(2)

# Create a summary table
missing_summary = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

# Show columns with missing values
missing_summary = missing_summary[
    missing_summary['Missing Values'] > 0
]

print(
    missing_summary.sort_values(
        by='Missing Percentage',
        ascending=False
    )
)


## 3. Identify laboratory orders and reported results

This section separates laboratory order events from reported result events and reviews missingness in fields relevant to the TAT analysis.

In [ ]:

# Columns important for LabPulse
important_columns = [
    'order_id',
    'test_name',
    'specimen_type',
    'collection_time',
    'processing_time',
    'result_time',
    'result_id',
    'result_status'
]

# Separate laboratory orders
lab_orders = df[
    df['event_type'] == 'lab_order_placed'
]

# Separate laboratory results
lab_results = df[
    df['event_type'] == 'lab_result_reported'
].copy()

# Check missing values in laboratory orders
order_missing = (
    lab_orders[important_columns]
    .isnull()
    .mean()
    .mul(100)
    .round(2)
)

print("LABORATORY ORDER MISSING VALUE PERCENTAGE")
print(order_missing)

# Check missing values in laboratory results
result_missing = (
    lab_results[important_columns]
    .isnull()
    .mean()
    .mul(100)
    .round(2)
)

print("\nLABORATORY RESULT MISSING VALUE PERCENTAGE")
print(result_missing)


In [ ]:
# Find laboratory result records
# where processing_time is missing

missing_processing = lab_results[
    lab_results['processing_time'].isnull()
]

print("Records with missing processing time:",
      len(missing_processing))

print("\nFirst 10 records:")
print(
    missing_processing[
        [
            'order_id',
            'test_name',
            'specimen_type',
            'collection_time',
            'processing_time',
            'result_time',
            'result_status'
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
missing_by_test = (
    missing_processing['test_name']
    .value_counts()
)

print("Missing processing timestamps by test:")
print(missing_by_test.to_string())

In [ ]:
# Count total records for each test
total_by_test = lab_results['test_name'].value_counts()

# Count missing processing timestamps for each test
missing_by_test = (
    missing_processing['test_name'].value_counts()
)

# Combine both counts into one DataFrame
test_missing_summary = pd.DataFrame({
    'Total Records': total_by_test,
    'Missing Records': missing_by_test
})

# Calculate missing percentage
test_missing_summary['Missing Percentage'] = (
    test_missing_summary['Missing Records']
    / test_missing_summary['Total Records'] * 100
).round(2)

# Sort from highest to lowest missing percentage
test_missing_summary = test_missing_summary.sort_values(
    by='Missing Percentage',
    ascending=False
)

print(test_missing_summary.to_string())

## 4. Prepare timestamps and calculate turnaround time

Collection and result timestamps are converted to pandas datetime values. TAT is calculated as the elapsed time from collection to result reporting, then expressed in minutes.

In [ ]:
# Check the current data types
print("Before conversion:")
print(lab_results[
    ['collection_time', 'result_time']
].dtypes)

# Convert timestamps into datetime format
lab_results['collection_time'] = pd.to_datetime(
    lab_results['collection_time'],
    errors='coerce'
)

lab_results['result_time'] = pd.to_datetime(
    lab_results['result_time'],
    errors='coerce'
)

# Check data types again
print("\nAfter conversion:")
print(lab_results[
    ['collection_time', 'result_time']
].dtypes)

In [ ]:
# Calculate turnaround time
lab_results['tat'] = (
    lab_results['result_time']
    - lab_results['collection_time']
)

# Convert TAT into minutes
lab_results['tat_minutes'] = (
    lab_results['tat'].dt.total_seconds() / 60
)

# Display sample records
print(
    lab_results[
        [
            'test_name',
            'collection_time',
            'result_time',
            'tat',
            'tat_minutes'
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
# Check for missing TAT values
print("Missing TAT values:",
      lab_results['tat_minutes'].isnull().sum())

# Check for negative TAT values
print("Negative TAT values:",
      (lab_results['tat_minutes'] < 0).sum())

# Get statistical summary of TAT
print("\nTAT Statistical Summary:")
print(lab_results['tat_minutes'].describe())

**Data-quality note:** TAT values depend on valid collection and result timestamps. Missing or invalid timestamps should be reviewed before interpreting downstream summaries.

## 5. Turnaround-time distribution and test-level analysis

This section explores the overall TAT distribution, compares tests, identifies long-TAT records, and examines variation between test categories.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    lab_results['tat_minutes'],
    bins=30,
    edgecolor='black'
)

plt.title('Distribution of Laboratory Turnaround Time')
plt.xlabel('Turnaround Time (Minutes)')
plt.ylabel('Number of Laboratory Results')

plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
tat_by_test = (
    lab_results.groupby('test_name')['tat_minutes']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .round(2)
    .sort_values(by='mean', ascending=False)
)

print(tat_by_test.to_string())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.barh(
    tat_by_test.index,
    tat_by_test['mean']
)

plt.title('Average Turnaround Time by Test Type')
plt.xlabel('Average TAT (Minutes)')
plt.ylabel('Laboratory Test')

plt.gca().invert_yaxis()

plt.grid(axis='x', alpha=0.3)
plt.show()

In [ ]:
slowest_results = lab_results.sort_values(
    by='tat_minutes',
    ascending=False
)

print(
    slowest_results[
        [
            'order_id',
            'test_name',
            'collection_time',
            'result_time',
            'tat_minutes'
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
# Filter records with TAT of 230 minutes or more
long_tat_records = lab_results[
    lab_results['tat_minutes'] >= 230
]

# Count these records
print("Records with TAT >= 230 minutes:",
      len(long_tat_records))

# Calculate their percentage
percentage = (
    len(long_tat_records) / len(lab_results)
) * 100

print("Percentage of total results:",
      round(percentage, 2), "%")

In [ ]:
long_tat_by_test = (
    long_tat_records['test_name']
    .value_counts()
)

print("Long TAT records by test type:")
print(long_tat_by_test.to_string())

In [ ]:
# Count long-TAT records for each test
long_tat_counts = (
    long_tat_records['test_name'].value_counts()
)

# Create a summary using total and long-TAT counts
long_tat_summary = pd.DataFrame({
    'Total Records': tat_by_test['count'],
    'Long TAT Records': long_tat_counts
})

# Calculate long-TAT percentage
long_tat_summary['Long TAT Percentage'] = (
    long_tat_summary['Long TAT Records']
    / long_tat_summary['Total Records'] * 100
).round(2)

# Sort by percentage from highest to lowest
long_tat_summary = long_tat_summary.sort_values(
    by='Long TAT Percentage',
    ascending=False
)

print(long_tat_summary.to_string())

In [ ]:
tat_variability = (
    lab_results.groupby('test_name')['tat_minutes']
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(2)
    .sort_values(by='std', ascending=False)
)

print(tat_variability.to_string())

In [ ]:
tat_variability['cv_percentage'] = (
    tat_variability['std']
    / tat_variability['mean']
) * 100

tat_variability['cv_percentage'] = (
    tat_variability['cv_percentage'].round(2)
)

tat_variability = tat_variability.sort_values(
    by='cv_percentage',
    ascending=False
)

print(tat_variability.to_string())

## 6. Daily trends and workload relationship

Daily sample volume and average TAT are compared over time. Correlation is descriptive and should not be interpreted as evidence that workload causes a change in TAT.

In [ ]:
lab_results['collection_date'] = (
    lab_results['collection_time'].dt.date
)

daily_tat = (
    lab_results.groupby('collection_date')['tat_minutes']
    .agg(['count', 'mean', 'median'])
    .round(2)
)

print(daily_tat.head(10).to_string())

In [ ]:
# 10 days with the highest average TAT
highest_tat_days = daily_tat.sort_values(
    by='mean',
    ascending=False
).head(10)

print("TOP 10 DAYS WITH HIGHEST AVERAGE TAT")
print(highest_tat_days.to_string())


# 10 days with the lowest average TAT
lowest_tat_days = daily_tat.sort_values(
    by='mean',
    ascending=True
).head(10)

print("\nTOP 10 DAYS WITH LOWEST AVERAGE TAT")
print(lowest_tat_days.to_string())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.plot(
    daily_tat.index,
    daily_tat['mean']
)

plt.title('Daily Average Laboratory Turnaround Time')
plt.xlabel('Collection Date')
plt.ylabel('Average TAT (Minutes)')

plt.xticks(rotation=45)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
daily_analysis = daily_tat.copy()

daily_analysis = daily_analysis.rename(
    columns={
        'count': 'daily_sample_volume',
        'mean': 'average_tat',
        'median': 'median_tat'
    }
)

print(daily_analysis.head(10).to_string())

In [ ]:
correlation = daily_analysis[
    ['daily_sample_volume', 'average_tat']
].corr()

print("Correlation between daily sample volume and average TAT:")
print(correlation)

**Interpretation:** Correlation describes the strength and direction of a linear association in this dataset. It does not establish causation.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    daily_analysis['daily_sample_volume'],
    daily_analysis['average_tat'],
    alpha=0.6
)

plt.title('Daily Sample Volume vs Average TAT')
plt.xlabel('Daily Number of Laboratory Results')
plt.ylabel('Average TAT (Minutes)')

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Hour-of-day analysis

The following analysis examines the distribution of collection volume and TAT across the 24 collection hours.

In [ ]:
lab_results['collection_hour'] = (
    lab_results['collection_time'].dt.hour
)

hourly_volume = (
    lab_results.groupby('collection_hour')
    .size()
    .reset_index(name='sample_count')
)

print(hourly_volume.to_string(index=False))

In [ ]:
peak_hours = hourly_volume.sort_values(
    by='sample_count',
    ascending=False
)

print(peak_hours.head(10).to_string(index=False))

In [ ]:
hourly_tat = (
    lab_results.groupby('collection_hour')['tat_minutes']
    .agg(['count', 'mean', 'median'])
    .round(2)
    .sort_values(by='mean', ascending=False)
)

print(hourly_tat.to_string())

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    hourly_volume['collection_hour'],
    hourly_volume['sample_count']
)

plt.title('Laboratory Sample Volume by Collection Hour')
plt.xlabel('Collection Hour')
plt.ylabel('Number of Samples')

plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
hourly_tat_plot = hourly_tat.sort_index()

plt.figure(figsize=(12, 5))

plt.plot(
    hourly_tat_plot.index,
    hourly_tat_plot['mean'],
    marker='o'
)

plt.title('Average Laboratory TAT by Collection Hour')
plt.xlabel('Collection Hour')
plt.ylabel('Average TAT (Minutes)')

plt.xticks(range(24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
hourly_correlation = hourly_tat[
    ['count', 'mean']
].corr()

print(hourly_correlation)

## 8. Weekday and weekend analysis

This section compares weekday/weekend patterns and explores variation by day of the week and test-day combinations.

In [ ]:
lab_results['day_of_week'] = (
    lab_results['collection_time'].dt.day_name()
)

print(
    lab_results['day_of_week']
    .value_counts()
)

In [ ]:
lab_results['day_type'] = (
    lab_results['collection_time'].dt.dayofweek
    .apply(
        lambda x: 'Weekend' if x >= 5 else 'Weekday'
    )
)

print(lab_results['day_type'].value_counts())

In [ ]:
weekday_comparison = (
    lab_results.groupby('day_type')['tat_minutes']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

print(weekday_comparison)

In [ ]:
weekday_tat = (
    lab_results.groupby('day_of_week')['tat_minutes']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

print(weekday_tat)

In [ ]:
weekday_tat_sorted = weekday_tat.sort_values(
    by='mean',
    ascending=False
)

print(weekday_tat_sorted)

In [ ]:
import matplotlib.pyplot as plt

weekday_tat_sorted = weekday_tat.sort_values(
    by='mean',
    ascending=False
)

plt.figure(figsize=(10, 5))

plt.bar(
    weekday_tat_sorted.index,
    weekday_tat_sorted['mean']
)

plt.title('Average Laboratory TAT by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average TAT (Minutes)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    weekday_tat_sorted.index,
    weekday_tat_sorted['mean']
)

plt.ylim(120, 132)

plt.title('Average Laboratory TAT by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average TAT (Minutes)')

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 9. Test and weekday combinations

These grouped summaries and heatmaps help compare TAT across test types and weekdays. Interpret small groups cautiously because sample counts can vary.

In [ ]:
test_weekday_tat = (
    lab_results
    .groupby(['test_name', 'day_of_week'])['tat_minutes']
    .agg(['count', 'mean', 'median'])
    .round(2)
)

test_weekday_tat

In [ ]:
# Highest average TAT combinations
highest_test_tat = test_weekday_tat.sort_values(
    by='mean',
    ascending=False
)

print("Top 10 highest TAT combinations:")
display(highest_test_tat.head(10))

In [ ]:
lowest_test_tat = test_weekday_tat.sort_values(
    by='mean',
    ascending=True
)

print("Top 10 lowest TAT combinations:")
display(lowest_test_tat.head(10))

In [ ]:
print("Top 10 lowest TAT combinations:")
display(lowest_test_tat.head(10))

In [ ]:
test_tat_range = (
    test_weekday_tat
    .groupby(level='test_name')['mean']
    .agg(['min', 'max'])
)

test_tat_range['tat_difference'] = (
    test_tat_range['max'] - test_tat_range['min']
)

test_tat_range = test_tat_range.sort_values(
    by='tat_difference',
    ascending=False
)

test_tat_range.round(2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert the grouped data into a matrix
heatmap_data = test_weekday_tat['mean'].unstack()

# Arrange weekdays in correct order
day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

heatmap_data = heatmap_data[day_order]

# Create heatmap
plt.figure(figsize=(12, 7))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5
)

plt.title('Average Laboratory TAT by Test and Weekday')
plt.xlabel('Day of Week')
plt.ylabel('Laboratory Test')
plt.tight_layout()
plt.show()

**Interpretation:** Heatmap colours summarize group averages. Always consider the sample count behind each test-day combination.

## 10. Exploratory delay analysis

A sample is classified as delayed when its TAT is **greater than or equal to 180 minutes**. This is an exploratory threshold for comparing patterns, not an official SLA.

The indicator is stored as a Boolean during analysis and converted to 0/1 in the exported CSV.

In [ ]:
# Create a delayed-sample indicator
lab_results['is_delayed'] = (
    lab_results['tat_minutes'] >= 180
)

# Calculate delay statistics by test
test_delay_analysis = (
    lab_results.groupby('test_name')['is_delayed']
    .agg(['count', 'sum', 'mean'])
)

# Convert proportion into percentage
test_delay_analysis['delay_percentage'] = (
    test_delay_analysis['mean'] * 100
)

# Sort by delay percentage
test_delay_analysis = test_delay_analysis.sort_values(
    by='delay_percentage',
    ascending=False
)

test_delay_analysis.round(2)

In [ ]:
total_samples = lab_results.shape[0]

total_delayed = lab_results['is_delayed'].sum()

overall_delay_percentage = (
    total_delayed / total_samples
) * 100

print("Total Samples:", total_samples)
print("Delayed Samples:", total_delayed)
print("Overall Delay Percentage:", round(overall_delay_percentage, 2), "%")

In [ ]:
weekday_delay_analysis = (
    lab_results.groupby('day_of_week')['is_delayed']
    .agg(['count', 'sum', 'mean'])
)

weekday_delay_analysis['delay_percentage'] = (
    weekday_delay_analysis['mean'] * 100
)

weekday_delay_analysis = weekday_delay_analysis.sort_values(
    by='delay_percentage',
    ascending=False
)

weekday_delay_analysis.round(2)

In [ ]:
plt.figure(figsize=(10, 5))

weekday_delay_sorted = weekday_delay_analysis.sort_values(
    by='delay_percentage',
    ascending=False
)

plt.bar(
    weekday_delay_sorted.index,
    weekday_delay_sorted['delay_percentage']
)

plt.title('Laboratory Delay Percentage by Weekday')
plt.xlabel('Day of Week')
plt.ylabel('Delayed Samples (%)')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
hourly_delay_analysis = (
    lab_results.groupby('collection_hour')['is_delayed']
    .agg(['count', 'sum', 'mean'])
)

hourly_delay_analysis['delay_percentage'] = (
    hourly_delay_analysis['mean'] * 100
)

hourly_delay_analysis = hourly_delay_analysis.sort_values(
    by='delay_percentage',
    ascending=False
)

hourly_delay_analysis.round(2)

In [ ]:
plt.figure(figsize=(12, 5))

hourly_delay_sorted = hourly_delay_analysis.sort_index()

plt.plot(
    hourly_delay_sorted.index,
    hourly_delay_sorted['delay_percentage'],
    marker='o'
)

plt.title('Laboratory Delay Percentage by Collection Hour')
plt.xlabel('Collection Hour (24-hour format)')
plt.ylabel('Delayed Samples (%)')

plt.xticks(range(24))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
weekday_hour_delay = (
    lab_results
    .groupby(['day_of_week', 'collection_hour'])['is_delayed']
    .mean()
    .mul(100)
    .unstack()
    .round(2)
)

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

weekday_hour_delay = weekday_hour_delay.reindex(day_order)

weekday_hour_delay

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18, 6))

sns.heatmap(
    weekday_hour_delay,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Delay Percentage (%)'}
)

plt.title('Laboratory Delay Percentage: Weekday vs Collection Hour')
plt.xlabel('Collection Hour')
plt.ylabel('Day of Week')

plt.tight_layout()
plt.show()

## 11. Statistical distribution and outlier exploration

Percentiles describe the TAT distribution, while the IQR method flags statistical outliers. An IQR outlier is not automatically an operational error or an unacceptable result.

In [ ]:
tat_percentiles = lab_results['tat_minutes'].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
)

print("TAT PERCENTILE ANALYSIS")
print(tat_percentiles)

In [ ]:
Q1 = lab_results['tat_minutes'].quantile(0.25)
Q3 = lab_results['tat_minutes'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = lab_results[
    (lab_results['tat_minutes'] < lower_limit) |
    (lab_results['tat_minutes'] > upper_limit)
]

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower Limit:", round(lower_limit, 2))
print("Upper Limit:", round(upper_limit, 2))

print("\nTotal Outliers:", len(outliers))
print("Outlier Percentage:", round(
    len(outliers) / len(lab_results) * 100, 2
), "%")

**Interpretation:** IQR-based outliers are statistical flags for further inspection, not proof of data errors.

## 12. Monthly TAT and workload analysis

Monthly summaries compare sample counts, average and median TAT, and the percentage of records at or above the exploratory delay threshold.

In [ ]:
lab_results['collection_month'] = (
    lab_results['collection_time'].dt.to_period('M')
)

In [ ]:
monthly_tat = (
    lab_results.groupby('collection_month')
    .agg(
        total_samples=('tat_minutes', 'count'),
        average_tat=('tat_minutes', 'mean'),
        median_tat=('tat_minutes', 'median'),
        delayed_samples=('is_delayed', 'sum'),
        delay_rate=('is_delayed', 'mean')
    )
)

monthly_tat['delay_percentage'] = (
    monthly_tat['delay_rate'] * 100
)

monthly_tat = monthly_tat.drop(
    columns='delay_rate'
).round(2)

print(monthly_tat.to_string())

In [ ]:
print("TOP 5 MONTHS — HIGHEST DELAY RATE")

print(
    monthly_tat.sort_values(
        by='delay_percentage',
        ascending=False
    ).head(5).to_string()
)

print("\nTOP 5 MONTHS — LOWEST DELAY RATE")

print(
    monthly_tat.sort_values(
        by='delay_percentage',
        ascending=True
    ).head(5).to_string()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.plot(
    monthly_tat.index.astype(str),
    monthly_tat['delay_percentage'],
    marker='o'
)

plt.axhline(
    y=15.59,
    linestyle='--',
    label='Overall Delay Rate (15.59%)'
)

plt.title('Monthly Laboratory Delay Percentage')
plt.xlabel('Collection Month')
plt.ylabel('Delayed Samples (%)')

plt.xticks(rotation=60)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
monthly_workload = (
    lab_results.groupby('collection_month')
    .agg(
        total_samples=('tat_minutes', 'count'),
        average_tat=('tat_minutes', 'mean'),
        delayed_samples=('is_delayed', 'sum'),
        delay_percentage=('is_delayed', 'mean')
    )
)

monthly_workload['delay_percentage'] *= 100

monthly_workload = monthly_workload.round(2)

print(monthly_workload.to_string())

In [ ]:
correlation = monthly_workload[
    ['total_samples', 'average_tat', 'delay_percentage']
].corr()

print(correlation.round(3))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.bar(
    monthly_workload.index.astype(str),
    monthly_workload['total_samples']
)

plt.title('Monthly Laboratory Sample Workload')
plt.xlabel('Month')
plt.ylabel('Total Samples')

plt.xticks(rotation=60)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(
    monthly_workload.sort_values(
        by='total_samples',
        ascending=False
    ).head(5).to_string()
)

## 13. Delay classification summaries and validation

This section labels records using the established delay flag, summarizes the classification, compares test-level patterns, and validates the threshold implementation.

In [ ]:
# Label records using the exploratory delay indicator
lab_results['delay_status'] = lab_results['is_delayed'].map({
    True: 'At or Above Threshold',
    False: 'Below Threshold'
})

print(lab_results['delay_status'].value_counts())

In [ ]:
# Summarize samples based on the exploratory delay threshold

delay_summary = (
    lab_results['delay_status']
    .value_counts()
    .to_frame('sample_count')
)

delay_summary['percentage'] = (
    delay_summary['sample_count']
    / len(lab_results) * 100
)

print(delay_summary.round(2))

In [ ]:
# Analyze delay classification by test type

test_delay_summary = (
    lab_results
    .groupby('test_name')
    .agg(
        total_samples=('event_id', 'count'),
        delayed_samples=('is_delayed', 'sum'),
        average_tat=('tat_minutes', 'mean')
    )
)

test_delay_summary['delay_percentage'] = (
    test_delay_summary['delayed_samples']
    / test_delay_summary['total_samples'] * 100
)

test_delay_summary = test_delay_summary.round(2)

print(test_delay_summary)

In [ ]:
print(
    test_delay_summary
    .sort_values(
        by='delay_percentage',
        ascending=False
    )
    .to_string()
)

## 14. Test-level percentiles and delay contribution

Percentiles provide a view of the upper tail of TAT by test. Delay contribution measures each test's share of all records classified as delayed; it differs from a test's own delay rate.

In [ ]:
test_percentiles = (
    lab_results.groupby('test_name')['tat_minutes']
    .quantile([0.50, 0.90, 0.95])
    .unstack()
)

test_percentiles.columns = [
    'P50_median',
    'P90',
    'P95'
]

test_percentiles = test_percentiles.round(2)

print(
    test_percentiles.sort_values(
        by='P95',
        ascending=False
    ).to_string()
)

In [ ]:
delay_contribution = (
    lab_results.groupby('test_name')
    .agg(
        total_samples=('tat_minutes', 'count'),
        delayed_samples=('is_delayed', 'sum')
    )
)

delay_contribution['delay_rate'] = (
    delay_contribution['delayed_samples']
    / delay_contribution['total_samples'] * 100
)

delay_contribution['contribution_percentage'] = (
    delay_contribution['delayed_samples']
    / lab_results['is_delayed'].sum() * 100
)

delay_contribution = delay_contribution.sort_values(
    by='delayed_samples',
    ascending=False
).round(2)

print(delay_contribution.to_string())

In [ ]:
# Validate the delay indicator against the agreed exploratory threshold
expected_delay_flag = lab_results['tat_minutes'].ge(180)

mismatched_records = (
    lab_results['is_delayed'].fillna(False) != expected_delay_flag.fillna(False)
).sum()

print("Mismatched Records:", int(mismatched_records))

In [ ]:
# Inspect any records whose delay classification does not match the threshold
mismatch_rows = lab_results.loc[
    lab_results['is_delayed'].fillna(False) !=
    lab_results['tat_minutes'].ge(180).fillna(False),
    ['test_name', 'tat_minutes', 'is_delayed']
]

print("Mismatched Records:", len(mismatch_rows))
display(mismatch_rows.head(20))

## 15. Overall summary and operational views

The following tables consolidate the main measures and provide additional test-level and time-based summaries for interpretation.

In [ ]:
# Overall laboratory performance summary

overall_summary = pd.DataFrame({
    'Metric': [
        'Total Samples',
        'Average TAT (Minutes)',
        'Median TAT (Minutes)',
        'Minimum TAT (Minutes)',
        'Maximum TAT (Minutes)',
        'Delayed Samples (>= 180 Minutes)',
        'Delay Percentage',
        'Below Threshold Samples',
        'Below Threshold Percentage'
    ],
    'Value': [
        len(lab_results),
        lab_results['tat_minutes'].mean(),
        lab_results['tat_minutes'].median(),
        lab_results['tat_minutes'].min(),
        lab_results['tat_minutes'].max(),
        lab_results['is_delayed'].sum(),
        lab_results['is_delayed'].mean() * 100,
        (~lab_results['is_delayed']).sum(),
        (~lab_results['is_delayed']).mean() * 100
    ]
})

overall_summary['Value'] = overall_summary['Value'].round(2)

print(overall_summary)

**Interpretation note:** All delay metrics in this notebook use the exploratory `TAT >= 180 minutes` definition. They should not be described as SLA compliance or breach rates.

In [ ]:
test_insights = (
    lab_results.groupby('test_name')
    .agg(
        total_samples=('tat_minutes', 'count'),
        delayed_samples=('is_delayed', 'sum')
    )
)

test_insights['delay_rate'] = (
    test_insights['delayed_samples']
    / test_insights['total_samples'] * 100
)

test_insights['delay_contribution'] = (
    test_insights['delayed_samples']
    / lab_results['is_delayed'].sum() * 100
)

test_insights = test_insights.sort_values(
    by='delayed_samples',
    ascending=False
).round(2)

print(test_insights.to_string())

In [ ]:
hourly_insights = (
    lab_results.groupby('collection_hour')
    .agg(
        total_samples=('tat_minutes', 'count'),
        delayed_samples=('is_delayed', 'sum'),
        average_tat=('tat_minutes', 'mean')
    )
)

hourly_insights['delay_rate'] = (
    hourly_insights['delayed_samples']
    / hourly_insights['total_samples'] * 100
)

hourly_insights = hourly_insights.sort_values(
    by='delay_rate',
    ascending=False
).round(2)

print(hourly_insights.to_string())

In [ ]:
weekday_hour_analysis = lab_results.pivot_table(
    index='day_of_week',
    columns='collection_hour',
    values='is_delayed',
    aggfunc='mean'
) * 100

weekday_hour_analysis = weekday_hour_analysis.round(2)

print(weekday_hour_analysis.to_string())

In [ ]:
weekday_hour_counts = lab_results.pivot_table(
    index='day_of_week',
    columns='collection_hour',
    values='tat_minutes',
    aggfunc='count',
    fill_value=0
)

weekday_hour_delays = lab_results.pivot_table(
    index='day_of_week',
    columns='collection_hour',
    values='is_delayed',
    aggfunc='sum',
    fill_value=0
)

print("Sample counts:")
print(weekday_hour_counts.to_string())

print("\nDelayed sample counts:")
print(weekday_hour_delays.to_string())

## 16. Export and verify the processed dataset

The final export is written to `../data/processed/labpulse_cleaned.csv`, relative to this notebook's location. The export stores `is_delayed` as an integer flag (0 = below threshold, 1 = at or above threshold).

In [ ]:
from pathlib import Path

# Create an export copy and store the delay indicator as 0/1
export_df = lab_results.copy()
export_df['is_delayed'] = export_df['tat_minutes'].ge(180).astype('int8')

# Save the processed dataset in a reproducible project-relative location
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "labpulse_cleaned.csv"
export_df.to_csv(csv_path, index=False)

# Verify the exported dataset
print(f"CSV exported successfully: {csv_path}")
print("Shape:", export_df.shape)
print("Delayed Samples:", int(export_df['is_delayed'].sum()))
print("Delay Percentage:", round(export_df['is_delayed'].mean() * 100, 2), "%")
print("\nDelay classification:")
print(export_df['is_delayed'].value_counts().sort_index())

In [ ]:
print("Exported dataset shape:", export_df.shape)

print("\nExported column names:")
print(export_df.columns.tolist())

In [ ]:
print("is_delayed dtype:", export_df['is_delayed'].dtype)
print("\nDelay flag counts:")
print(export_df['is_delayed'].value_counts().sort_index())
print("\nUnique delay flag values:", sorted(export_df['is_delayed'].unique()))

In [ ]:
# Read the exported file back to verify that it is accessible and correctly saved
reloaded_df = pd.read_csv(csv_path)

print("Reloaded CSV shape:", reloaded_df.shape)
print("Reloaded delayed samples:", int(reloaded_df['is_delayed'].sum()))
print(
    "Export row-count check:",
    "PASS" if len(reloaded_df) == len(export_df) else "CHECK"
)

## 17. Key findings and conclusion

Use the outputs generated above to summarize the strongest patterns observed in this synthetic dataset. Keep conclusions descriptive and evidence-based.

### Suggested reporting checklist
- Report the number of laboratory result records included in the analysis.
- State the average and median TAT, with units in minutes.
- Identify tests or collection periods with comparatively higher TAT or delay percentages.
- Distinguish a group's **delay rate** from its **contribution to all delayed records**.
- Mention that workload-TAT correlations are descriptive and do not establish causation.

### Limitations
- The dataset is synthetic; patterns may not reflect real laboratory operations.
- The 180-minute threshold is exploratory and is not an official SLA.
- Aggregated comparisons can be affected by differences in group sizes and test mix.
- Observational patterns do not establish causal relationships.

### Future scope
- Add confidence intervals or significance testing for selected comparisons.
- Investigate whether test mix and other available factors explain TAT variation.
- Extend the analysis with validated operational definitions if real, authorized data becomes available.
